In [2]:
from pyspark.sql import functions as F

landing_path = "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing"


StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 4, Finished, Available, Finished, False)

In [3]:
df_facilities = (
    spark.read
    .option("header", "true")
    .csv(landing_path + "/facilities.csv")
)

display(df_facilities)

print(f"Source records: {df_facilities.count()}")

df_facilities.printSchema()

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8679384b-a29a-41fb-9f68-1ed88ba7f253)

Source records: 50
root
 |-- facility_id: string (nullable = true)
 |-- facility_name: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- dock_doors: string (nullable = true)
 |-- operating_hours: string (nullable = true)



In [4]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 6, Finished, Available, Finished, False)

In [5]:
facility_schema = StructType([
    StructField("facility_id", StringType(), True),
    StructField("facility_name", StringType(), True),
    StructField("facility_type", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("dock_doors", IntegerType(), True),
    StructField("operating_hours", StringType(), True)
])

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 7, Finished, Available, Finished, False)

In [6]:
df_facilities = (
    spark.read
    .option("header", "true")
    .schema(facility_schema)
    .csv(landing_path + "/facilities.csv")
)

display(df_facilities)

print(f"Source records: {df_facilities.count()}")

df_facilities.printSchema()

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c6f8389e-cfd7-4fd5-a6a8-5e3cc4443136)

Source records: 50
root
 |-- facility_id: string (nullable = true)
 |-- facility_name: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- dock_doors: integer (nullable = true)
 |-- operating_hours: string (nullable = true)



In [7]:
source_count = df_facilities.count()
print(f"Source records: {source_count}")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 9, Finished, Available, Finished, False)

Source records: 50


In [8]:
null_facility_ids = (
    df_facilities
    .filter(F.col("facility_id").isNull())
    .count()
)

print(f"NULL facility IDs: {null_facility_ids}")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 10, Finished, Available, Finished, False)

NULL facility IDs: 0


In [9]:
duplicate_facility_ids = (
    df_facilities
    .groupBy("facility_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate facility IDs: {duplicate_facility_ids}")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 11, Finished, Available, Finished, False)

Duplicate facility IDs: 0


In [10]:
# Latitude must be between -90 and 90
invalid_latitude = (
    df_facilities
    .filter(
        (F.col("latitude") < -90) |
        (F.col("latitude") > 90)
    )
    .count()
)

print(f"Invalid latitude: {invalid_latitude}")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 12, Finished, Available, Finished, False)

Invalid latitude: 0


In [11]:
# Longitude must be between -180 and 180
invalid_longitude = (
    df_facilities
    .filter(
        (F.col("longitude") < -180) |
        (F.col("longitude") > 180)
    )
    .count()
)

print(f"Invalid longitude: {invalid_longitude}")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 13, Finished, Available, Finished, False)

Invalid longitude: 0


In [12]:
# Dock doors cannot be negative
invalid_dock_doors = (
    df_facilities
    .filter(F.col("dock_doors") < 0)
    .count()
)

print(f"Invalid dock door count: {invalid_dock_doors}")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 14, Finished, Available, Finished, False)

Invalid dock door count: 0


In [13]:
df_facilities = (
    df_facilities
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("facilities.csv"))
)

df_facilities.createOrReplaceTempView("facilities_source")

print("Facility source data prepared for Bronze layer.")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 15, Finished, Available, Finished, False)

Facility source data prepared for Bronze layer.


In [14]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_facilities (
    facility_id STRING,
    facility_name STRING,
    facility_type STRING,
    city STRING,
    state STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    dock_doors INT,
    operating_hours STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("bronze_facilities table is ready.")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 16, Finished, Available, Finished, False)

bronze_facilities table is ready.


In [15]:
result = spark.sql("""
MERGE INTO bronze_facilities AS target
USING facilities_source AS source

ON target.facility_id = source.facility_id

WHEN MATCHED THEN
    UPDATE SET
        target.facility_name = source.facility_name,
        target.facility_type = source.facility_type,
        target.city = source.city,
        target.state = source.state,
        target.latitude = source.latitude,
        target.longitude = source.longitude,
        target.dock_doors = source.dock_doors,
        target.operating_hours = source.operating_hours,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        facility_id,
        facility_name,
        facility_type,
        city,
        state,
        latitude,
        longitude,
        dock_doors,
        operating_hours,
        ingestion_timestamp,
        source_file
    )
    VALUES (
        source.facility_id,
        source.facility_name,
        source.facility_type,
        source.city,
        source.state,
        source.latitude,
        source.longitude,
        source.dock_doors,
        source.operating_hours,
        source.ingestion_timestamp,
        source.source_file
    )
""")

display(result)

print("Facility Bronze MERGE completed successfully.")

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, daf7b2b4-b614-414d-9d27-b3b0dcd8495e)

Facility Bronze MERGE completed successfully.


In [16]:
bronze_count = spark.sql("""
SELECT COUNT(*) AS count
FROM bronze_facilities
""").collect()[0]["count"]

print(f"Bronze facility records: {bronze_count}")

display(
    spark.sql("""
    SELECT *
    FROM bronze_facilities
    LIMIT 10
    """)
)

StatementMeta(, db9a2d8e-b6dd-4a33-a4b2-6e39dad25ebc, 18, Finished, Available, Finished, False)

Bronze facility records: 50


SynapseWidget(Synapse.DataFrame, 5cf03171-1cfb-4dd1-9f02-2fade615f8a5)